# Session 4 — BART Summarization
**Task:** Generate a short summary from a long article  
**Model:** `facebook/bart-large-cnn` (pretrained on CNN/DailyMail)  
**Dataset:** CNN/DailyMail  
**Metric:** ROUGE-1, ROUGE-2, ROUGE-L

---
### Key difference from BERT sessions
- BERT = encoder-only → classification/span extraction  
- BART = encoder-decoder → **generates** new tokens (abstractive summarization)  
- Input: article → Output: generated summary (not extracted span)
- We use a pretrained checkpoint already fine-tuned on news — so inference works out of the box, fine-tuning just improves it further

## Step 1 — Imports & Config

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset

DEVICE      = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME  = "facebook/bart-large-cnn"
MAX_INPUT   = 512    # max article tokens
MAX_TARGET  = 128    # max summary tokens
BATCH_SIZE  = 4      # bart-large is big — keep batch small
EPOCHS      = 2
LR          = 3e-5
TRAIN_SIZE  = 1000
VAL_SIZE    = 200
SAVE_DIR    = "../../models/05_transformers/bart_summarization"

print(f"Device: {DEVICE}")

## Step 2 — Load & Inspect Dataset

In [ ]:
raw = load_dataset("cnn_dailymail", "3.0.0")
print(raw)

ex = raw["train"][0]
print("\n--- Article (first 300 chars) ---")
print(ex["article"][:300])
print("\n--- Highlights (summary) ---")
print(ex["highlights"])

## Step 3 — Tokenizer
For seq2seq: tokenize article as input, tokenize highlights as labels (decoder targets).  
Labels use `-100` for padding — cross-entropy ignores those positions.

In [ ]:
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)

ex = raw["train"][0]
enc_input  = tokenizer(ex["article"],    max_length=MAX_INPUT,  truncation=True)
enc_target = tokenizer(ex["highlights"], max_length=MAX_TARGET, truncation=True)

print(f"Article tokens:  {len(enc_input['input_ids'])}")
print(f"Summary tokens:  {len(enc_target['input_ids'])}")

## Step 4 — Dataset

In [ ]:
class SummarizationDataset(Dataset):
    def __init__(self, hf_split, tokenizer):
        self.data      = hf_split
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex = self.data[idx]

        model_inputs = self.tokenizer(
            ex["article"],
            max_length=MAX_INPUT,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        labels = self.tokenizer(
            ex["highlights"],
            max_length=MAX_TARGET,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )["input_ids"].squeeze(0)

        # Replace padding token id with -100 so loss ignores padding
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids":      model_inputs["input_ids"].squeeze(0),
            "attention_mask": model_inputs["attention_mask"].squeeze(0),
            "labels":         labels,
        }

train_raw = raw["train"].select(range(TRAIN_SIZE))
val_raw   = raw["validation"].select(range(VAL_SIZE))

train_ds = SummarizationDataset(train_raw, tokenizer)
val_ds   = SummarizationDataset(val_raw,   tokenizer)

item = train_ds[0]
print("input_ids shape:", item["input_ids"].shape)
print("labels shape:   ", item["labels"].shape)

## Step 5 — Model, Optimizer, Scheduler

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

model = BartForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"BART params: {total_params:,}")   # ~400M — much bigger than BERT

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)
print(f"Device: {DEVICE}")

## Step 6 — Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_loss(model, loader):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)
            outputs  = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
    return total_loss / len(loader)


for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler)
    val_loss   = evaluate_loss(model, val_loader)
    print(f"Epoch {epoch}/{EPOCHS} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f}")

## Step 7 — Save & Inference

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")


def summarize(text, model, tokenizer, max_new_tokens=128):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_INPUT)
    input_ids = inputs["input_ids"].to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            num_beams=4,          # beam search — better than greedy
            length_penalty=2.0,   # penalise short summaries
            early_stopping=True,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


article = """
Scientists have discovered a new species of deep-sea fish in the Pacific Ocean at a depth of
6,000 meters. The fish, named Pseudoliparis swirei, was found in the Mariana Trench and
appears to be one of the deepest-living fish ever recorded. Researchers used a specially
designed deep-sea submersible to capture footage and collect samples. The discovery provides
new insights into how life adapts to extreme pressure and cold temperatures. The findings
were published in the journal Zootaxa.
"""
print("Summary:", summarize(article, model, tokenizer))